# Chapter 14 — Advanced Panel Data: Fixed Effects & Random Effects
## University of Neuchâtel — Python for Econometrics

Topics:
1. Panel data structure and the unobserved effects model
2. Pooled OLS (ignores individual effects)
3. Fixed Effects (Within estimator) — manual demeaning
4. First Differences estimator
5. Random Effects estimator
6. Hausman test: FE vs RE

**Format:** One cell per step. Manual calculation first, then linearmodels/statsmodels, then `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import wooldridge as woo
import os
try:
    from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS as lmPooledOLS
    HAS_LINEARMODELS = True
except ImportError:
    HAS_LINEARMODELS = False
    print('linearmodels not installed — will use manual + statsmodels only')

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Panel Data & the Unobserved Effects Model

**Panel data** = same $N$ individuals observed over $T$ time periods.

**Model:**

$$y_{it} = \beta_1 x_{it} + a_i + u_{it}$$

- $a_i$: **unobserved individual effect** (fixed over time, e.g. ability, work ethic)
- $u_{it}$: idiosyncratic error

**Problem:** If $Cov(a_i, x_{it}) \neq 0$, pooled OLS is biased.

**Solution:** Fixed Effects — eliminate $a_i$ by demeaning or first differencing.

## 2 — Load Data: `wagepan`

In [ ]:
df = woo.dataWoo('wagepan')
print('Shape:', df.shape)
print(df[['nr','year','lwage','educ','exper','union','married','black','hisp']].head(12))

In [ ]:
print('Panel structure: obs per person')
obs_per = df.groupby('nr').size()
print(obs_per.describe())
print(f'Number of individuals: {df["nr"].nunique()}')
print(f'Number of time periods: {df["year"].nunique()} ({df["year"].min()}-{df["year"].max()})')

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
sample_ids = df['nr'].unique()[:8]
for nrid in sample_ids:
    sub = df[df['nr']==nrid]
    ax.plot(sub['year'], sub['lwage'], marker='o', linewidth=1, alpha=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('log(wage)')
ax.set_title('lwage over time for 8 individuals')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch14_01_panel_trends.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3 — Pooled OLS (Ignores Panel Structure)

### Step 3.1 — Manual OLS: lwage ~ educ + exper + union + married

In [ ]:
n = len(df)
X_pool = np.column_stack([
    np.ones(n),
    df['educ'].values,
    df['exper'].values,
    df['union'].values,
    df['married'].values
])
y_arr = df['lwage'].values
beta_pool = np.linalg.solve(X_pool.T @ X_pool, X_pool.T @ y_arr)
labels_pool = ['const','educ','exper','union','married']
print('Pooled OLS (manual):')
for lbl, b in zip(labels_pool, beta_pool):
    print(f'  {lbl:10s}: {b:.4f}')

### Step 3.2 — Statsmodels Pooled OLS

In [ ]:
res_pool = smf.ols('lwage ~ educ + exper + union + married', data=df).fit()
print(res_pool.summary())

### Step 3.3 — Compare

In [ ]:
sm_p = res_pool.params
print('Pooled OLS comparison:')
for lbl, b_man in zip(labels_pool, beta_pool):
    match = np.isclose(b_man, sm_p[lbl])
    print(f'  {lbl:10s}: manual={b_man:.4f}, sm={sm_p[lbl]:.4f}, match={match}')

## 4 — Fixed Effects (Within Estimator)

**Idea:** Demean all variables by individual mean to remove $a_i$:

$$\ddot{y}_{it} = y_{it} - \bar{y}_i, \quad \ddot{x}_{it} = x_{it} - \bar{x}_i$$

$$\ddot{y}_{it} = \beta_1 \ddot{x}_{it} + \ddot{u}_{it}$$

**Key property:** Time-invariant variables (like `educ`, `black`) are eliminated (their $\ddot{x} = 0$).

### Step 4.1 — Compute individual means for each variable

In [ ]:
fe_vars = ['lwage','exper','union','married']
for v in fe_vars:
    df[v+'_mean_i'] = df.groupby('nr')[v].transform('mean')
print('Dataset after adding individual means (first 8 rows):')
print(df[['nr','year','lwage','lwage_mean_i','exper','exper_mean_i']].head(8))

### Step 4.2 — Demean variables

In [ ]:
for v in fe_vars:
    df[v+'_dm'] = df[v] - df[v+'_mean_i']
print('Demeaned variables (first 8 rows):')
print(df[['nr','year','lwage_dm','exper_dm','union_dm','married_dm']].head(8))

### Step 4.3 — Manual OLS on demeaned data

In [ ]:
X_fe = np.column_stack([
    df['exper_dm'].values,
    df['union_dm'].values,
    df['married_dm'].values
])
y_fe = df['lwage_dm'].values
# No constant — demeaning removes it
beta_fe_manual = np.linalg.solve(X_fe.T @ X_fe, X_fe.T @ y_fe)
labels_fe = ['exper','union','married']
print('Fixed Effects (manual demeaning):')
for lbl, b in zip(labels_fe, beta_fe_manual):
    print(f'  {lbl:10s}: {b:.4f}')

### Step 4.4 — FE via linearmodels PanelOLS

In [ ]:
if HAS_LINEARMODELS:
    df_panel = df.set_index(['nr','year'])
    res_fe = PanelOLS.from_formula(
        'lwage ~ exper + union + married + EntityEffects',
        data=df_panel
    ).fit()
    print(res_fe.summary)
else:
    res_fe_sm = smf.ols('lwage_dm ~ exper_dm + union_dm + married_dm - 1', data=df).fit()
    print(res_fe_sm.summary())

### Step 4.5 — Compare FE manual vs PanelOLS

In [ ]:
if HAS_LINEARMODELS:
    fe_params = res_fe.params
    print('FE comparison:')
    for lbl, b_man in zip(labels_fe, beta_fe_manual):
        b_sm = fe_params[lbl]
        match = np.isclose(b_man, b_sm)
        print(f'  {lbl:10s}: manual={b_man:.4f}, lm={b_sm:.4f}, match={match}')
else:
    fe_sm = smf.ols('lwage_dm ~ exper_dm + union_dm + married_dm - 1', data=df).fit()
    for lbl, b_man, b_sm_name in zip(labels_fe, beta_fe_manual, ['exper_dm','union_dm','married_dm']):
        b_sm = fe_sm.params[b_sm_name]
        print(f'  {lbl:10s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={np.isclose(b_man,b_sm)}')

## 5 — First Differences Estimator

**Alternative to demeaning:** Take first differences to eliminate $a_i$:

$$\Delta y_{it} = y_{it} - y_{i,t-1} = \beta_1 \Delta x_{it} + \Delta u_{it}$$

Note: FD and FE are equivalent when $T=2$; differ when $T>2$.

### Step 5.1 — Compute first differences manually

In [ ]:
df_sorted = df.sort_values(['nr','year']).copy()
fd_vars = ['lwage','exper','union','married']
for v in fd_vars:
    df_sorted['d_'+v] = df_sorted.groupby('nr')[v].diff()
print('First differences (first 12 rows):')
print(df_sorted[['nr','year','lwage','d_lwage','exper','d_exper']].head(12))
print('Note: NaN for first obs of each person')

### Step 5.2 — Manual OLS on first differences

In [ ]:
df_fd = df_sorted[['d_lwage','d_exper','d_union','d_married']].dropna()
print(f'FD dataset shape: {df_fd.shape}')
X_fd = np.column_stack([
    np.ones(len(df_fd)),
    df_fd['d_exper'].values,
    df_fd['d_union'].values,
    df_fd['d_married'].values
])
y_fd = df_fd['d_lwage'].values
beta_fd = np.linalg.solve(X_fd.T @ X_fd, X_fd.T @ y_fd)
labels_fd = ['const','d_exper','d_union','d_married']
print('FD OLS (manual):')
for lbl, b in zip(labels_fd, beta_fd):
    print(f'  {lbl:12s}: {b:.4f}')

### Step 5.3 — FD via statsmodels

In [ ]:
res_fd = smf.ols('d_lwage ~ d_exper + d_union + d_married', data=df_fd).fit()
print(res_fd.summary())

### Step 5.4 — Compare FD manual vs statsmodels

In [ ]:
sm_fd = res_fd.params
print('FD comparison:')
for lbl, b_man, sm_name in zip(labels_fd, beta_fd, ['Intercept','d_exper','d_union','d_married']):
    b_sm = sm_fd[sm_name]
    print(f'  {lbl:12s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={np.isclose(b_man,b_sm)}')

## 6 — Summary: Pooled OLS vs FE vs FD

In [ ]:
print('Summary: union coefficient across estimators')
print(f'  Pooled OLS: {res_pool.params["union"]:.4f}')
if HAS_LINEARMODELS:
    print(f'  Fixed Effects: {res_fe.params["union"]:.4f}')
else:
    print(f'  Fixed Effects (manual): {beta_fe_manual[1]:.4f}')
print(f'  First Differences: {res_fd.params["d_union"]:.4f}')
print('FE/FD remove time-invariant unobserved heterogeneity')
print('If Pooled OLS union coeff >> FE, suggests positive selection into unions')

In [ ]:
sumtab = pd.DataFrame({
    'Estimator': ['Pooled OLS','Fixed Effects','First Differences'],
    'exper': [
        res_pool.params['exper'],
        beta_fe_manual[0],
        res_fd.params['d_exper']
    ],
    'union': [
        res_pool.params['union'],
        beta_fe_manual[1],
        res_fd.params['d_union']
    ],
    'married': [
        res_pool.params['married'],
        beta_fe_manual[2],
        res_fd.params['d_married']
    ]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch14_panel_comparison.csv'), index=False)
print('Saved table_ch14_panel_comparison.csv')